# IGDB Data Collection Notebook

**Goal:** Build a comprehensive dataset of PS4/PS5 games with metadata, covers, and screenshots for ML projects.

**Output:**
- Structured DataFrame (parquet + CSV) with ~10K games
- Cover art images (~2-4 GB)
- Screenshots, up to 5 per game (~15-25 GB)

**Expected runtime:** 1-3 hours total. Has resume capability if interrupted.

**Before you start:**
1. Register an app at https://dev.twitch.tv/console/apps
2. Get your Client ID and Client Secret
3. Fill them in Cell 3 below


## Cell 1: Install dependencies
Run this once. Skip if already installed.

In [ ]:
!pip install requests pandas tqdm pillow pyarrow matplotlib python-dotenv -q

## Cell 2: Imports and setup

In [ ]:
import requests
import pandas as pd
import json
import time
import os
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import logging
import math

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## Cell 3: Credentials and directories
**Fill in your Twitch Developer credentials below.**

Get them from https://dev.twitch.tv/console/apps (free)

In [ ]:
# Get these from https://dev.twitch.tv/console/apps
CLIENT_ID = 'YOUR_CLIENT_ID_HERE'
CLIENT_SECRET = 'YOUR_CLIENT_SECRET_HERE'

# Directories
BASE_DIR = Path('igdb_dataset')
DATA_DIR = BASE_DIR / 'data'
COVERS_DIR = BASE_DIR / 'covers'
SCREENSHOTS_DIR = BASE_DIR / 'screenshots'

for d in [DATA_DIR, COVERS_DIR, SCREENSHOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Base directory: {BASE_DIR.absolute()}')

## Cell 4: Authentication
Exchanges your credentials for an OAuth access token (valid for ~60 days).

In [ ]:
def get_access_token(client_id, client_secret):
    url = 'https://id.twitch.tv/oauth2/token'
    params = {
        'client_id': client_id,
        'client_secret': client_secret,
        'grant_type': 'client_credentials'
    }
    response = requests.post(url, params=params)
    response.raise_for_status()
    token_data = response.json()
    logger.info(f"Got access token, expires in {token_data['expires_in']} seconds")
    return token_data['access_token']

ACCESS_TOKEN = get_access_token(CLIENT_ID, CLIENT_SECRET)
HEADERS = {
    'Client-ID': CLIENT_ID,
    'Authorization': f'Bearer {ACCESS_TOKEN}',
    'Accept': 'application/json'
}

## Cell 5: Core API function
Wraps the IGDB API with retry logic for rate limiting and transient errors.

In [ ]:
def igdb_query(endpoint, query, max_retries=3):
    """Execute an Apicalypse query against IGDB API."""
    url = f'https://api.igdb.com/v4/{endpoint}'
    
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=HEADERS, data=query, timeout=30)
            
            if response.status_code == 429:
                wait_time = 2 ** attempt
                logger.warning(f'Rate limited. Waiting {wait_time}s...')
                time.sleep(wait_time)
                continue
            
            response.raise_for_status()
            return response.json()
        
        except requests.exceptions.RequestException as e:
            logger.error(f'Attempt {attempt + 1} failed: {e}')
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
    
    return []

## Cell 6: Test the connection
Run this to verify everything works before starting the big collection.

In [ ]:
test_query = 'fields name, rating; where total_rating_count > 100; limit 5;'
test_result = igdb_query('games', test_query)

print('Connection test - first 5 games:')
for game in test_result:
    print(f"  - {game.get('name')} (rating: {game.get('rating', 'N/A')})")

## Cell 7: Define the main query
**Platform IDs reference:**
- PS5: 167
- PS4: 48
- PS3: 9
- Xbox Series X|S: 169
- Xbox One: 49
- PC: 6
- Nintendo Switch: 130

Change `PLATFORMS` below if you want a different set.

In [ ]:
PLATFORMS = [48, 167]  # PS4 and PS5
PLATFORMS_STR = ','.join(map(str, PLATFORMS))

FIELDS = '''
    id, name, slug, summary, storyline, 
    rating, rating_count, total_rating, total_rating_count,
    aggregated_rating, aggregated_rating_count,
    first_release_date, category, status,
    genres.name, themes.name, game_modes.name, player_perspectives.name,
    keywords.name,
    cover.image_id, cover.width, cover.height,
    screenshots.image_id, artworks.image_id,
    involved_companies.company.name, involved_companies.developer, involved_companies.publisher,
    platforms.name, platforms.id,
    similar_games,
    franchises.name, collections.name,
    age_ratings.rating, age_ratings.category,
    game_engines.name,
    videos.video_id
'''

print(f'Will collect data for platforms: {PLATFORMS}')

## Cell 8: Collect all games (the main run)
This is the slow part. Expect 15-30 minutes for 10K games.

**Has resume capability** - if it crashes, just re-run this cell and it picks up where it left off.

In [ ]:
def collect_games(platforms_str, batch_size=500, save_every=10):
    progress_file = DATA_DIR / 'collection_progress.json'
    games_file = DATA_DIR / 'games_raw.json'
    
    if progress_file.exists():
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        offset = progress['next_offset']
        logger.info(f'Resuming from offset {offset}')
        
        with open(games_file, 'r') as f:
            all_games = json.load(f)
    else:
        offset = 0
        all_games = []
    
    batch_num = 0
    pbar = tqdm(desc='Collecting games', unit=' games')
    pbar.update(len(all_games))
    
    while True:
        query = f'''
        fields {FIELDS};
        where platforms = ({platforms_str}) 
            & total_rating_count > 3
            & cover != null
            & category = 0;
        limit {batch_size};
        offset {offset};
        sort first_release_date asc;
        '''
        
        try:
            batch = igdb_query('games', query)
        except Exception as e:
            logger.error(f'Failed at offset {offset}: {e}')
            break
        
        if not batch:
            logger.info('No more games to fetch')
            break
        
        all_games.extend(batch)
        pbar.update(len(batch))
        offset += batch_size
        batch_num += 1
        
        if batch_num % save_every == 0:
            with open(games_file, 'w') as f:
                json.dump(all_games, f)
            with open(progress_file, 'w') as f:
                json.dump({'next_offset': offset}, f)
            logger.info(f'Saved {len(all_games)} games to disk')
        
        time.sleep(0.35)  # Rate limit: 4 req/sec max
    
    pbar.close()
    
    with open(games_file, 'w') as f:
        json.dump(all_games, f)
    
    progress_file.unlink(missing_ok=True)
    
    logger.info(f'Collection complete! Total games: {len(all_games)}')
    return all_games

games_data = collect_games(PLATFORMS_STR)

## Cell 9: Convert to a clean DataFrame
Flattens the nested JSON into a proper tabular format.

In [ ]:
def normalize_games(games):
    rows = []
    
    for g in games:
        genres = [x['name'] for x in g.get('genres', [])]
        themes = [x['name'] for x in g.get('themes', [])]
        modes = [x['name'] for x in g.get('game_modes', [])]
        perspectives = [x['name'] for x in g.get('player_perspectives', [])]
        keywords = [x['name'] for x in g.get('keywords', [])]
        
        developers = []
        publishers = []
        for ic in g.get('involved_companies', []):
            company_name = ic.get('company', {}).get('name', '')
            if ic.get('developer'):
                developers.append(company_name)
            if ic.get('publisher'):
                publishers.append(company_name)
        
        platforms = [x['name'] for x in g.get('platforms', [])]
        platform_ids = [x['id'] for x in g.get('platforms', [])]
        
        cover_id = g.get('cover', {}).get('image_id') if g.get('cover') else None
        screenshot_ids = [s['image_id'] for s in g.get('screenshots', [])]
        artwork_ids = [a['image_id'] for a in g.get('artworks', [])]
        
        release_date = None
        if g.get('first_release_date'):
            release_date = datetime.fromtimestamp(g['first_release_date'])
        
        rating = g.get('total_rating')
        rating_count = g.get('total_rating_count', 0)
        if rating and rating_count:
            weighted_score = rating * math.log1p(rating_count) / 10
        else:
            weighted_score = None
        
        rows.append({
            'id': g['id'],
            'name': g.get('name'),
            'slug': g.get('slug'),
            'summary': g.get('summary'),
            'storyline': g.get('storyline'),
            'release_date': release_date,
            'release_year': release_date.year if release_date else None,
            'rating': g.get('rating'),
            'rating_count': g.get('rating_count', 0),
            'total_rating': g.get('total_rating'),
            'total_rating_count': g.get('total_rating_count', 0),
            'aggregated_rating': g.get('aggregated_rating'),
            'aggregated_rating_count': g.get('aggregated_rating_count', 0),
            'weighted_score': weighted_score,
            'genres': genres,
            'themes': themes,
            'game_modes': modes,
            'player_perspectives': perspectives,
            'keywords': keywords[:20],
            'developers': developers,
            'publishers': publishers,
            'platforms': platforms,
            'platform_ids': platform_ids,
            'on_ps5': 167 in platform_ids,
            'on_ps4': 48 in platform_ids,
            'cover_id': cover_id,
            'screenshot_ids': screenshot_ids,
            'artwork_ids': artwork_ids,
            'num_screenshots': len(screenshot_ids),
            'similar_games': g.get('similar_games', []),
            'franchises': [f['name'] for f in g.get('franchises', [])],
            'collections': [c['name'] for c in g.get('collections', [])],
        })
    
    return pd.DataFrame(rows)

df = normalize_games(games_data)
print(f'Dataset shape: {df.shape}')
print(f'\nSample stats:')
print(f'  Games with cover: {df["cover_id"].notna().sum()}')
print(f'  Games with screenshots: {(df["num_screenshots"] > 0).sum()}')
print(f'  Games with rating: {df["total_rating"].notna().sum()}')
print(f'  Average screenshots per game: {df["num_screenshots"].mean():.1f}')

df.to_parquet(DATA_DIR / 'games.parquet')
df.to_csv(DATA_DIR / 'games.csv', index=False)
logger.info(f'Saved structured dataset: {len(df)} games')

## Cell 10: Define image download functions

**Image sizes available:**
- `cover_small`: 90x128
- `cover_big`: 264x374 (recommended for covers)
- `screenshot_med`: 569x320
- `screenshot_big`: 889x500 (recommended for screenshots)
- `screenshot_huge`: 1280x720
- `720p`: 1280x720
- `1080p`: 1920x1080 (large, use sparingly)

In [ ]:
def get_image_url(image_id, size='cover_big'):
    return f'https://images.igdb.com/igdb/image/upload/t_{size}/{image_id}.jpg'

def download_image(url, filepath, retries=3):
    if filepath.exists():
        return True
    
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=15, stream=True)
            response.raise_for_status()
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(8192):
                    f.write(chunk)
            return True
        except Exception as e:
            if attempt == retries - 1:
                logger.warning(f'Failed to download {url}: {e}')
                return False
            time.sleep(1)
    return False

## Cell 11: Download covers
Parallel download of all cover art. ~20-40 min depending on internet speed.

In [ ]:
def download_covers(df, size='cover_big', max_workers=10):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    games_with_covers = df[df['cover_id'].notna()][['id', 'cover_id']]
    
    tasks = []
    for _, row in games_with_covers.iterrows():
        url = get_image_url(row['cover_id'], size)
        filepath = COVERS_DIR / f"{row['id']}.jpg"
        tasks.append((url, filepath))
    
    successful = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_image, url, fp): (url, fp) 
                   for url, fp in tasks}
        
        for future in tqdm(as_completed(futures), total=len(tasks), 
                          desc='Downloading covers'):
            if future.result():
                successful += 1
    
    logger.info(f'Downloaded {successful}/{len(tasks)} covers')

download_covers(df, size='cover_big')

## Cell 12: Download screenshots
Downloads up to 5 screenshots per game. This is the heaviest step (~1-2 hours).

**Tip:** Reduce `max_per_game` to 3 or use `screenshot_med` size if disk space is tight.

In [ ]:
def download_screenshots(df, size='screenshot_big', max_per_game=5, max_workers=10):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    tasks = []
    for _, row in df.iterrows():
        if not row['screenshot_ids']:
            continue
        
        game_dir = SCREENSHOTS_DIR / str(row['id'])
        game_dir.mkdir(exist_ok=True)
        
        for i, ss_id in enumerate(row['screenshot_ids'][:max_per_game]):
            url = get_image_url(ss_id, size)
            filepath = game_dir / f'{i:02d}_{ss_id}.jpg'
            tasks.append((url, filepath))
    
    successful = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_image, url, fp): (url, fp)
                   for url, fp in tasks}
        
        for future in tqdm(as_completed(futures), total=len(tasks),
                          desc='Downloading screenshots'):
            if future.result():
                successful += 1
    
    logger.info(f'Downloaded {successful}/{len(tasks)} screenshots')

download_screenshots(df, size='screenshot_big', max_per_game=5)

## Cell 13: Dataset summary and statistics
Verify everything worked and explore what you have.

In [ ]:
print('='*60)
print('DATASET SUMMARY')
print('='*60)
print(f'Total games collected: {len(df)}')
print(f'PS5 games: {df["on_ps5"].sum()}')
print(f'PS4 games: {df["on_ps4"].sum()}')
print(f'Cross-platform (both): {(df["on_ps5"] & df["on_ps4"]).sum()}')

print(f'\nRating distribution:')
print(df['total_rating'].describe())

print(f'\nTop 10 genres:')
all_genres = [g for genres in df['genres'] for g in genres]
print(pd.Series(all_genres).value_counts().head(10))

print(f'\nTop 10 publishers:')
all_pubs = [p for pubs in df['publishers'] for p in pubs]
print(pd.Series(all_pubs).value_counts().head(10))

print(f'\nRelease year distribution (last 15 years):')
print(df['release_year'].value_counts().sort_index().tail(15))

n_covers = len(list(COVERS_DIR.glob('*.jpg')))
n_screenshots = sum(1 for _ in SCREENSHOTS_DIR.rglob('*.jpg'))
print(f'\nImages downloaded:')
print(f'  Covers: {n_covers}')
print(f'  Screenshots: {n_screenshots}')

def get_dir_size(path):
    total = sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())
    return total / (1024**2)

print(f'\nDisk usage:')
print(f'  Covers: {get_dir_size(COVERS_DIR):.1f} MB')
print(f'  Screenshots: {get_dir_size(SCREENSHOTS_DIR):.1f} MB')
print(f'  Total: {get_dir_size(BASE_DIR):.1f} MB')

## Cell 14: Visualization - sample covers
Sanity check that the images downloaded correctly.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 5, figsize=(15, 8))
sample = df[df['cover_id'].notna()].sample(10, random_state=42)

for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img_path = COVERS_DIR / f"{row['id']}.jpg"
    if img_path.exists():
        img = Image.open(img_path)
        ax.imshow(img)
        title = f"{row['name'][:20]}\nRating: {row['total_rating']:.0f}" if row['total_rating'] else row['name'][:20]
        ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(DATA_DIR / 'sample_covers.png', dpi=100)
plt.show()

## ✅ Done!

Your dataset is ready. To reload later:

```python
import pandas as pd
df = pd.read_parquet('igdb_dataset/data/games.parquet')
```

**Next steps:**
1. Feature engineering on text + categorical fields
2. Train baseline models on cover images for rating prediction
3. Build similarity matrices for recommendation system